<div class="alert alert-info" markdown="1">

#### Homework 6 Supplemental Notebook

# Inverses and LU Factorization

### Math 124: Vectors, Matrices, and Applications, Fall 2026

<small><a style="text-decoration: none" href="https://math124.org">math124.org</a></small>

</div>

### Instructions

Most homeworks and some labs will have a Jupyter Notebook, like this one, containing Python code that supplements our understanding of the relevant mathematical ideas of the week.

---

Open this notebook in Google Colab. Instructions on how to use Google Colab are at [math124.org/running-code](https://math124.org/running-code).

**Most importantly, make sure to click "Copy to Drive" and rename the notebook before proceeding!**

---

Together, the tasks in this notebook contribute to your score for Homework 6, Problems 5 and 8. You won't need to submit this notebook anywhere. Follow the orange submission instructions in the homework PDF, and upload your completed PDF to Pensive.

## Setup

---

Run the cell below to import the libraries we will use.

In [ ]:
import numpy as np
from scipy.linalg import lu, solve_triangular
from IPython.display import Audio, display
import base64
import io

np.set_printoptions(precision=3, suppress=True)

## Problem 5: Inverses in NumPy (12 points)

---

Use `np.linalg.inv(M)` to compute the inverse of a matrix `M`. Remember that `@` multiplies matrices.

Use the cells below for Problem 5. Enter the matrices from the homework yourself.

Parts a), b), and c) are worth 4 points each.

In [ ]:
# M1


In [ ]:
# M2


In [ ]:
# M3


In [ ]:
# M4


In [ ]:
# M5


In [ ]:
# M6


Use the space below for the matrix products in part b). Include your answers to part c) in your homework PDF. Remember that you can create as many new cells as you'd like.

## Problem 8: Separating Songs (18 points)

---

Five speakers are playing five different songs at the same time, **one song per speaker**:

- **Speaker 1:** “Mr. Brightside” – The Killers.
- **Speaker 2:** “Dancing Queen” – ABBA.
- **Speaker 3:** “Patient Zero” – Taylor Swift.
- **Speaker 4:** “Choosin’ Texas” – Ella Langley.
- **Speaker 5:** “The Victors” – University of Michigan.

The speakers stand in a line, with five microphones arranged around them in a circle. We know the song titles, but **do not** have access to the original song files – all we have are what our microphones recorded. Our goal is to recover the audio originally depicted by each speaker – i.e., the original songs – from these mixtures.

Digital audio stores amplitudes at regularly spaced instants. At 8,000 Hz, there are 8,000 measurements per second, so a four-second clip contains 32,000 numbers.

At a particular instant, let $x_1,x_2,x_3,x_4,x_5$ be the five speaker amplitudes and $b_1,b_2,b_3,b_4,b_5$ be the five microphone amplitudes. An amplitude can be positive, negative, or zero. The weights are:

| | Speaker 1<br><small>(Song 1)</small> | Speaker 2<br><small>(Song 2)</small> | Speaker 3<br><small>(Song 3)</small> | Speaker 4<br><small>(Song 4)</small> | Speaker 5<br><small>(Song 5)</small> |
|---|---:|---:|---:|---:|---:|
| Microphone 1 | 0.9 | 0.8 | 0.6 | 0.5 | 0.7 |
| Microphone 2 | 0.7 | 0.9 | 0.8 | 0.7 | 0.6 |
| Microphone 3 | 0.5 | 0.6 | 0.8 | 0.9 | 0.8 |
| Microphone 4 | 0.8 | 0.7 | 0.9 | 0.8 | 0.6 |
| Microphone 5 | 0.6 | 0.8 | 0.7 | 0.6 | 0.9 |

For example, $b_1=0.9x_1+0.8x_2+0.6x_3+0.5x_4+0.7x_5$.

**Assume these weights stay fixed throughout the problem.** Complete **Problem 8, part a) (4 points) on paper**: write the five equations and the matrix equation $A\vec x=\vec b$.

## Load and listen to the microphone recordings

---

Run the supplied cell below without editing it. It loads two recording sessions, each containing five simultaneous microphone recordings, each four seconds long. The second session uses different excerpts of the same five songs, with the same speaker assignments.

`B1` and `B2` are both matrices with shape `(5, 32000)`. A **row** contains one microphone's recording over time. A **column** contains all five microphone amplitudes at one instant: one column of `B1` (or `B2`) is one vector $\vec b$ that we can use to solve for $\vec x$ in $A \vec x = \vec b$.

The recordings use 8,000 samples per second and are four seconds long, hence each recording has 32,000 $\vec b$'s that we need to extract original sounds from.

Run the cell below to load the recordings and listen to the noisy mixtures from all five microphones in each session. Later, in parts c) and d), you will play the recovered songs.

In [ ]:
#@title Load the microphone recordings and audio players { display-mode: "form" }
_recordings_data = """..."""
with np.load(io.BytesIO(base64.b64decode(_recordings_data)), allow_pickle=False) as data:
    B1 = data["B1"].astype(float)
    B2 = data["B2"].astype(float)
SAMPLE_RATE = 8000
print("Session 1 shape:", B1.shape)
print("Session 2 shape:", B2.shape)

def play_channels(recordings, label):
    """Play each row separately, preserving the amplitudes."""
    recordings = np.asarray(recordings)
    for i in range(recordings.shape[0]):
        print(f"{label} {i + 1}")
        # Standard-rate playback; the original arrays used for solving stay unchanged.
        from scipy.signal import resample_poly
        playback = resample_poly(recordings[i], 441, 80)
        display(Audio(playback, rate=44100, normalize=False))

play_channels(B1, "Session 1: microphone mixture")
print('\n\n\n')
play_channels(B2, "Session 2: microphone mixture")

## Problem 8, part b): Solve for one instant (3 points)

---

Enter the matrix $A$ from part a). The rest of the code for this part is provided; run it and inspect the results.

The supplied code later in this part selects one column of `B1`, corresponding to a time about 1.54 seconds into the first recording session. These five microphone amplitudes form the vector $\vec b$ at that instant.

In [ ]:
A = ...

To perform an LU factorization, we will use the `scipy` package, a closely-related package to `numpy`. SciPy's LU function uses a different permutation convention than we learned about in class. The call

```python
Q, L, U = lu(A)
P = Q.T
```

returns matrices satisfying $A=QLU$. To get our convention $PA=LU$, set **`P = Q.T`**. Here `.T` transposes a matrix, exchanging rows and columns. For a permutation matrix, this gives its inverse.

Below, we verify that $PA=LU$ by displaying `P @ A` and `L @ U`. Then use $L$ and $U$ to solve for the five speaker amplitudes, $\vec x$, for the one instant $\vec b$ selected below. Why does the first right-hand side become $P\vec b$? Starting with $A\vec x=\vec b$, multiply both sides by $P$:

$$PA\vec x=P\vec b \quad\Longrightarrow\quad LU\vec x=P\vec b.$$

As we've seen in class, we'll first solve $L\vec y=P\vec b$ for $\vec y$, then solve $U\vec x=\vec y$ for $\vec x$. `solve_triangular` is a function built into `scipy` that knows how to perform forward substitution and back substitution to solve systems of equations involving triangular matrices.


```python
y = solve_triangular(L, P @ b, lower=True)
x = solve_triangular(U, y, lower=False)
```
Run the next cell to compute the factors and display the two products. Compare them to verify that $PA=LU$.

In [ ]:
Q, L, U = lu(A)
P = Q.T

print("P @ A:")
print(P @ A)
print("L @ U:")
print(L @ U)

The function below performs the two triangular solves for you. First it solves $L\vec y=P\vec b$, then it solves $U\vec x=\vec y$ and returns $\vec x$. Run this cell without editing it.

In [ ]:
def solve_with_factors(P, L, U, b):
    y = solve_triangular(L, P @ b, lower=True)
    x = solve_triangular(U, y, lower=False)
    return x

In [ ]:
sample_index = 12345
b = B1[:, sample_index]

x = solve_with_factors(P, L, U, b)
print("Microphone amplitudes b:")
print(b)
print("Recovered speaker amplitudes x:")
print(x)

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include a screenshot of the microphone amplitudes and the recovered speaker amplitudes for the one instant provided above.</p>

## Problem 8, part c): Recover the five song clips (3 points)

---

Recall, in the first four-second clip, there are a total of 32,000 “instants,” i.e., 32,000 vectors $\vec b$, each of which describes the amplitude recorded by each of the five microphones.

Using the LU decomposition from part b), solve all these systems together to recover the five song clips.

Fortunately, `solve_triangular` can solve for many right-hand sides at once: **pass the whole matrix `B1` in place of the single vector `b`.** The supplied function from part b) works for either shape, with no changes!

The result, `X1`, should have shape `(5, 32000)`. Its columns contain the recovered speaker amplitudes at each instant. Its rows contain the five separate song clips. In matrix notation, we are solving $AX_1=B_1$.

Fill in the cell below – we tell you what to do a few sentences above. It should use the same $P$, $L$, and $U$ to recover all 32,000 instants at once and stores the result in `X1`; you do not need to write a loop or compute new factors. Then play the recovered clips and listen to how they differ from the microphone mixtures above.

In [ ]:
X1 = ...
print("Recovered recordings shape:", X1.shape)

After running the recovery cell above, run the supplied cell below to listen to the five recovered songs, one per speaker.

In [ ]:
play_channels(X1, "Session 1: recovered speaker")

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include a screenshot of the code that starts with `X1 = ` and of the five audio players that appear afterwards.</p>

## Problem 8, part d): Reuse the factors for a second recording session (2 points)

---

In the second session, each speaker plays a different excerpt of its song.

Your job is to recover the five clips from `B2` using the same factors. Check your result and listen to the recovered clips.

In [ ]:
X2 = ...

Once again, run the cell below to listen to the five recovered songs, one per speaker. If you'd like, you can listen to the original noisy microphone recordings in the loading section at the start of this section.

In [ ]:
play_channels(X2, "Session 2: recovered speaker")

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include a screenshot of your code and check for the second recording session.</p>

## Problem 8, part e): Questions about the systems (6 points)

---

Return to the PDF; we have a few questions you'd like you to answer there.

## Finish line 🏁

Congrats on finishing the programming component of the homework! The main assignment tells you what to include from here in your submission to Pensive.